In [ ]:
import sys, os, time
import numpy as np
sys.path.append(os.path.abspath('..'))
from drone_env.drone_sim import RoomDroneEnv
import pybullet as p

env = RoomDroneEnv(gui=True)
obs, info = env.reset()
p.resetDebugVisualizerCamera(cameraDistance=4.0, cameraYaw=45, cameraPitch=-30, cameraTargetPosition=[0, 0, 1])

# Flight Choreography: Each movement will last half a second (120 steps).
choreography = [
    ("1. HOVER (Warm-up)", np.array([0.0, 0.0, 0.0, 0.0], dtype=np.float32)),
    ("2. AGGRESSIVE FORWARD (Max Pitch)", np.array([0.8, 0.0, 0.0, 0.0], dtype=np.float32)),
    ("3. SUDDEN BRAKE (Max Backward Pitch)", np.array([-0.8, 0.0, 0.0, 0.0], dtype=np.float32)),
    ("4. AGGRESSIVE RIGHT STRAFE (Roll)", np.array([0.0, 0.8, 0.0, 0.0], dtype=np.float32)),
    ("5. MIXER (Forward + Left + Turn)", np.array([0.6, -0.6, 0.8, 0.0], dtype=np.float32)),
    ("6. FREE FALL (Motors Off)", np.array([0.0, 0.0, 0.0, -1.0], dtype=np.float32)),
    ("7. PANIC RECOVERY (Full Throttle Hover)", np.array([0.0, 0.0, 0.0, 0.5], dtype=np.float32))
]

print("DEATH TEST INITIATED...\n")

survived = True
for name, action in choreography:
    print(f">>> Executing {name}...")
    for _ in range(120): # Half a second
        obs, reward, terminated, truncated, info = env.step(action)
        time.sleep(1./240.) # Real-time so you can observe it
        
        if terminated:
            print(f"\n[FATAL CRASH] Drone died during the '{name}' phase!")
            survived = False
            break
    if not survived:
        break

if survived:
    print("\n[PERFECT] DRONE SURVIVED THE ENTIRE STRESS TEST! PHYSICS ENGINE IS FLAWLESS.")
    
# Wait 3 seconds to observe the result
time.sleep(3)
env.close()